In [3]:
import ROOT
import os
import math
from pathlib import Path
ROOT.gStyle.SetOptStat(0)

#build signal dataframe
sig_chain = ROOT.TChain("AnalysisMiniTree")
sig_chain.Add("signal_ggF.root")
sig_chain.Add("signal_VBF.root")
df_sig = ROOT.RDataFrame(sig_chain)

#your cuts, used for both signal and each background below
selection = "n_b_jet == 0 && mZ_cut > 0 && l1_charge != l2_charge"

base_dir = Path(".")
tree_name = "AnalysisMiniTree"
exclude_files = {"signal_ggF.root", "signal_VBF.root"}

background_files = sorted([
    p for p in base_dir.glob("*.root")
    if p.name not in exclude_files
])

if not background_files:
    print(f"No background files found in {base_dir}")

# Branches to skip (IDs/flags/weights, etc.)
skip_vars = {"dsid", "eventNumber", "weight"}

# Fixed ranges for known variable types, checked by substring match
fixed_ranges = [
    ("phi", (-math.pi, math.pi)),
    ("eta", (-4, 4)),
    ("dR", (0, 2 * math.pi)),
]

def get_fixed_range(var):
    for key, rng in fixed_ranges:
        if key in var:
            return rng
    return None

# Signal dataframe (apply same selection if available)
df_sig_plot = df_sig.Filter(selection) if "selection" in globals() else df_sig
sig_cols = set(str(name) for name in df_sig_plot.GetColumnNames())

total_saved = 0

for bkg_file in background_files:
    bkg_name = bkg_file.stem
    print(f"\nProcessing background: {bkg_name}")

    f_bkg = ROOT.TFile.Open(str(bkg_file))
    if not f_bkg or f_bkg.IsZombie():
        print(f"  skipping {bkg_name}: cannot open file")
        continue

    t_bkg = f_bkg.Get(tree_name)
    if not t_bkg:
        print(f"  skipping {bkg_name}: missing {tree_name}")
        f_bkg.Close()
        continue

    df_bkg = ROOT.RDataFrame(t_bkg)
    df_bkg_plot = df_bkg.Filter(selection) if "selection" in globals() else df_bkg
    bkg_cols = set(str(name) for name in df_bkg_plot.GetColumnNames())

    common_vars = sorted(sig_cols.intersection(bkg_cols))

    out_dir = Path("plots") / bkg_name
    out_dir.mkdir(parents=True, exist_ok=True)

    saved_for_bkg = 0

    for var in common_vars:
        if var in skip_vars:
            continue

        try:
            fixed = get_fixed_range(var)

            if fixed is not None:
                x_min, x_max = fixed
            else:
                sig_min = float(df_sig_plot.Min(var).GetValue())
                sig_max = float(df_sig_plot.Max(var).GetValue())
                bkg_min = float(df_bkg_plot.Min(var).GetValue())
                bkg_max = float(df_bkg_plot.Max(var).GetValue())

                if not all(math.isfinite(v) for v in (sig_min, sig_max, bkg_min, bkg_max)):
                    continue

                x_min = min(sig_min, bkg_min)
                x_max = max(sig_max, bkg_max)

                if x_min == x_max:
                    continue

                x_pad = (x_max - x_min) * 0.05
                x_min -= x_pad
                x_max += x_pad

            h_sig_ptr = df_sig_plot.Histo1D(
                (f"h_sig_{bkg_name}_{var}", f"{var};{var};Normalized to unit area", 50, x_min, x_max),
                var,
                "weight",
            )
            h_bkg_ptr = df_bkg_plot.Histo1D(
                (f"h_bkg_{bkg_name}_{var}", f"{var};{var};Normalized to unit area", 50, x_min, x_max),
                var,
                "weight",
            )

            h_sig = h_sig_ptr.GetValue()
            h_bkg = h_bkg_ptr.GetValue()

            sig_int = h_sig.Integral()
            bkg_int = h_bkg.Integral()
            if sig_int <= 0 or bkg_int <= 0:
                continue

            h_sig.Scale(1.0 / sig_int)
            h_bkg.Scale(1.0 / bkg_int)

            c = ROOT.TCanvas(f"c_{bkg_name}_{var}", "", 800, 600)
            c.SetLeftMargin(0.16)
            c.SetRightMargin(0.05)
            c.SetBottomMargin(0.14)
            c.SetTopMargin(0.08)

            h_bkg.SetLineColor(ROOT.kPink + 2)
            h_bkg.SetLineWidth(2)
            h_sig.SetLineColor(ROOT.kBlue + 1)
            h_sig.SetLineWidth(2)

            h_bkg.GetXaxis().SetTitleOffset(1.2)
            h_bkg.GetYaxis().SetTitleOffset(1.7)
            h_sig.GetXaxis().SetTitleOffset(1.2)
            h_sig.GetYaxis().SetTitleOffset(1.7)

            y_max = max(h_sig.GetMaximum(), h_bkg.GetMaximum()) * 1.25
            if h_sig.GetMaximum() >= h_bkg.GetMaximum():
                h_first, h_second = h_sig, h_bkg
            else:
                h_first, h_second = h_bkg, h_sig

            h_first.SetMinimum(0.0)
            h_first.SetMaximum(y_max)
            h_first.Draw("hist")
            h_second.Draw("hist same")

            leg = ROOT.TLegend(0.62, 0.75, 0.88, 0.88)
            leg.AddEntry(h_sig, "Signal", "l")
            leg.AddEntry(h_bkg, bkg_name, "l")
            leg.Draw()

            safe_var = var.replace("/", "_").replace(" ", "_")
            c.SaveAs(str(out_dir / f"{safe_var}.png"))

            saved_for_bkg += 1
            total_saved += 1

        except Exception as e:
            print(f"  skipping {var}: {e}")

    print(f"  saved {saved_for_bkg} plots in {out_dir}")
    f_bkg.Close()

print(f"\nDone. Total plots saved: {total_saved}")



Processing background: VVV


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_VVV_HT
Info in <TCanvas::Print>: png file plots/VVV/HT.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_VVV_HT_all
Info in <TCanvas::Print>: png file plots/VVV/HT_all.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_VVV_HT_jet
Info in <TCanvas::Print>: png file plots/VVV/HT_jet.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_VVV_HT_lep
Info in <TCanvas::Print>: png file plots/VVV/HT_lep.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_VVV_HT_lepMET
Info in <TCanvas::Print>: png file plots/VVV/HT_lepMET.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_VVV_HT_tau
Info in <TCanvas::Print>: png file plots/VVV/HT_tau.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_VVV_SumPt_l1j
I

  saved 99 plots in plots/VVV

Processing background: Vgamma


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_Vgamma_HT_all
Info in <TCanvas::Print>: png file plots/Vgamma/HT_all.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_Vgamma_HT_jet
Info in <TCanvas::Print>: png file plots/Vgamma/HT_jet.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_Vgamma_HT_lep
Info in <TCanvas::Print>: png file plots/Vgamma/HT_lep.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_Vgamma_HT_lepMET
Info in <TCanvas::Print>: png file plots/Vgamma/HT_lepMET.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_Vgamma_HT_tau
Info in <TCanvas::Print>: png file plots/Vgamma/HT_tau.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_Vgamma_SumPt_l1j
Info in <TCanvas::Print>: png file plots/Vgamma/SumPt_l1j.png has been created
Warning in <TCanvas::Constructor>:

  saved 99 plots in plots/Vgamma

Processing background: Wjets


...

  saved 99 plots in plots/singleH

Processing background: tops


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_tops_HT
Info in <TCanvas::Print>: png file plots/tops/HT.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_tops_HT_all
Info in <TCanvas::Print>: png file plots/tops/HT_all.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_tops_HT_jet
Info in <TCanvas::Print>: png file plots/tops/HT_jet.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_tops_HT_lep
Info in <TCanvas::Print>: png file plots/tops/HT_lep.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_tops_HT_lepMET
Info in <TCanvas::Print>: png file plots/tops/HT_lepMET.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_tops_HT_tau
Info in <TCanvas::Print>: png file plots/tops/HT_tau.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_top

  saved 99 plots in plots/tops

Processing background: ttbar


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_ttbar_HT_all
Info in <TCanvas::Print>: png file plots/ttbar/HT_all.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_ttbar_HT_jet
Info in <TCanvas::Print>: png file plots/ttbar/HT_jet.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_ttbar_HT_lep
Info in <TCanvas::Print>: png file plots/ttbar/HT_lep.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_ttbar_HT_lepMET
Info in <TCanvas::Print>: png file plots/ttbar/HT_lepMET.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_ttbar_HT_tau
Info in <TCanvas::Print>: png file plots/ttbar/HT_tau.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_ttbar_SumPt_l1j
Info in <TCanvas::Print>: png file plots/ttbar/SumPt_l1j.png has been created
Warning in <TCanvas::Constructor>: Deleting ca

  saved 92 plots in plots/ttbar

Done. Total plots saved: 780
